# SSD_AIMO3 Thesis Validation Engine

This notebook is the primary engine for initial Colab experimentation.

Its job is not to produce optimistic demos. Its job is to help us **(in)validate the thesis** under a disciplined A0 -> A1 -> A5 ladder:

- `A0`: strongest fair frozen baseline
- `A1`: naive SSD self-distillation
- `A5`: tropical reranking over generated eval traces

Success means a decision-ready answer, not just runnable code.

## Zero-edit starter

This notebook defaults to `EXPERIMENT_MODE = "auto"` and boots a public real benchmark path without requiring hand-built manifests.

By default, `auto` prepares a GSM8K-based real run with a small public math model so the first serious Colab pass can produce thesis-facing evidence.

Set `EXPERIMENT_MODE = "starter"` only when you want the fast fixture-backed harness check instead.

## Method discipline

Before running anything, keep the evaluation contract fixed:

1. The primary metric is exact final-answer accuracy.
2. Baseline and student must use the same extraction rules.
3. A1 only counts as useful if it beats the best fair A0 baseline.
4. A5 only counts as useful if it improves over matched non-tropical evaluation on the same generated traces.
5. Any gain that comes from extraction bugs, prompt drift, or incomparable budgets is not a scientific win.

Use the final comparison outputs to answer:

- Did A1 beat A0?
- Did A5 add value beyond A1?
- Are the discordant-pair counts large enough to justify more budget?
- What failure modes dominated the losses?

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import shlex
from pathlib import Path


PROJECT_ROOT = Path("/content/SSD_AIMO3")
RUN_NAME = "colab_notebook_session"
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
BUNDLE_DIR = RUN_ROOT / "bundle"


def run(*args: str, cwd: Path | None = None) -> None:
    cmd = [str(arg) for arg in args]
    print("$", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd or PROJECT_ROOT),
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        rendered = shlex.join(cmd)
        raise RuntimeError(f"Command failed with exit code {return_code}: {rendered}")


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def update_env(name: str, value) -> None:
    if value is None or value == "":
        os.environ.pop(name, None)
        return
    os.environ[name] = str(value)


def flatten_scalars(prefix: str, mapping) -> dict:
    payload = {}

    def visit(value, path):
        if isinstance(value, dict):
            for key, item in value.items():
                visit(item, path + [str(key)])
            return
        key = ".".join([part for part in path if part])
        if not key:
            return
        if isinstance(value, (str, int, float, bool)) or value is None:
            payload[key] = value
        else:
            payload[key] = json.dumps(value, sort_keys=True)

    visit(mapping, [prefix] if prefix else [])
    return payload


def log_wandb_scalars(run, prefix: str, mapping) -> None:
    if run is None:
        return
    payload = flatten_scalars(prefix, mapping)
    if payload:
        run.log(payload)


def configure_wandb(secret_name: str, project: str, entity: str, mode: str, tags, notes: str | None):
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    api_key = os.environ.get(secret_name)
    if not api_key and userdata is not None:
        try:
            api_key = userdata.get(secret_name)
        except Exception:
            api_key = None
    if not api_key:
        raise RuntimeError(f"Missing W&B API key. Add {secret_name} to Colab secrets.")

    import wandb

    wandb_dir = RUN_ROOT / "wandb"
    wandb_dir.mkdir(parents=True, exist_ok=True)
    update_env("WANDB_API_KEY", api_key)
    update_env("WANDB_PROJECT", project)
    update_env("WANDB_ENTITY", entity or None)
    update_env("WANDB_MODE", mode)
    update_env("WANDB_DIR", wandb_dir)
    update_env("SSD_AIMO3_WANDB_ENABLED", "1")
    update_env("SSD_AIMO3_WANDB_PROJECT", project)
    update_env("SSD_AIMO3_WANDB_ENTITY", entity or None)
    update_env("SSD_AIMO3_WANDB_MODE", mode)
    update_env("SSD_AIMO3_WANDB_GROUP", RUN_NAME)
    update_env("SSD_AIMO3_WANDB_RUN_PREFIX", RUN_NAME)
    update_env("SSD_AIMO3_WANDB_TAGS", ",".join(tags))
    update_env("SSD_AIMO3_WANDB_NOTES", notes or None)
    update_env("SSD_AIMO3_WANDB_DIR", wandb_dir)
    update_env("SSD_AIMO3_WANDB_LOG_ARTIFACTS", "1")
    update_env("SSD_AIMO3_WANDB_ARTIFACT_MAX_FILE_SIZE_MB", WANDB_ARTIFACT_MAX_FILE_SIZE_MB)

    wandb.login(key=api_key, relogin=True)
    return wandb.init(
        project=project,
        entity=entity or None,
        name=f"{RUN_NAME}-session",
        group=RUN_NAME,
        job_type="colab_session",
        tags=list(tags),
        notes=notes or None,
        dir=str(wandb_dir),
        reinit=True,
        config={
            "experiment_mode": RESOLVED_EXPERIMENT_MODE,
            "requested_experiment_mode": EXPERIMENT_MODE,
            "run_name": RUN_NAME,
            "model_id": MODEL_ID,
            "prompt_manifest_jsonl": str(PROMPT_MANIFEST_JSONL),
            "eval_prompt_manifest_jsonl": str(EVAL_PROMPT_MANIFEST_JSONL),
            "problem_metadata_jsonl": str(PROBLEM_METADATA_JSONL),
            "num_eval_samples": NUM_EVAL_SAMPLES,
        },
    )


def ensure_repo(repo_url: str, repo_ref: str) -> None:
    if (PROJECT_ROOT / ".git").exists():
        print(f"Using existing repo at {PROJECT_ROOT}")
        return
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"{PROJECT_ROOT} exists but is not a git checkout. Remove it or point PROJECT_ROOT elsewhere.")
    run("git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(PROJECT_ROOT), cwd=Path("/content"))
    print(f"Cloned {repo_url}@{repo_ref} -> {PROJECT_ROOT}")


def is_placeholder(value) -> bool:
    return isinstance(value, str) and value.startswith("REPLACE_WITH_")


def collect_real_mode_issues() -> list[str]:
    issues = []
    if REAL_MODE_PRESET:
        if REAL_MODE_PRESET != "public_gsm8k_qwen25_math_1p5b":
            issues.append(f"Unsupported REAL_MODE_PRESET: {REAL_MODE_PRESET}")
        if not str(MODEL_ID).strip() or is_placeholder(MODEL_ID):
            issues.append("MODEL_ID could not be resolved from the selected public preset.")
        if int(REAL_PRESET_TRAIN_LIMIT) <= 0:
            issues.append("REAL_PRESET_TRAIN_LIMIT must be positive.")
        if int(REAL_PRESET_EVAL_LIMIT) <= 0:
            issues.append("REAL_PRESET_EVAL_LIMIT must be positive.")
        return issues
    if is_placeholder(MODEL_ID):
        issues.append("MODEL_ID is still set to a placeholder.")
    if RAW_INPUT_PATH and not Path(RAW_INPUT_PATH).exists():
        issues.append(f"RAW_INPUT_PATH does not exist: {RAW_INPUT_PATH}")
    if not RAW_INPUT_PATH:
        for label, path in (
            ("PROMPT_MANIFEST_JSONL", PROMPT_MANIFEST_JSONL),
            ("EVAL_PROMPT_MANIFEST_JSONL", EVAL_PROMPT_MANIFEST_JSONL),
            ("PROBLEM_METADATA_JSONL", PROBLEM_METADATA_JSONL),
        ):
            if not Path(path).exists():
                issues.append(f"{label} does not exist: {path}")
    return issues


NOTEBOOK_WANDB_RUN = None

PROJECT_ROOT

In [ ]:
# Default: self-contained auto mode.
# Auto now prefers a public real benchmark so the notebook can produce genuine A0 -> A1 -> A5 evidence without manual manifests.
# Set EXPERIMENT_MODE to "starter" for a fast fixture-backed harness check.

EXPERIMENT_MODE = "auto"  # "auto", "starter", or "real"
REPO_URL = "https://github.com/fyremael/SSD_AIMO3.git"
REPO_REF = "main"

assert EXPERIMENT_MODE in {"auto", "starter", "real"}

REAL_MODE_PRESET = "public_gsm8k_qwen25_math_1p5b"  # set to "" to bring your own model and manifests
REAL_PRESET_TRAIN_LIMIT = 256
REAL_PRESET_EVAL_LIMIT = 64
REAL_PRESET_SEED = 17

MODEL_ID = "REPLACE_WITH_HF_MODEL_ID"
RAW_INPUT_PATH = ""  # optional: CSV / JSON / JSONL problem-level source
RAW_INPUT_FORMAT = "auto"
PROBLEM_ID_FIELD = "problem_id"
PROMPT_FIELD = "prompt_text"
ANSWER_FIELD = "gold_answer"
TOPIC_FIELD = "topic"
DIFFICULTY_FIELD = "difficulty"
TAGS_FIELD = "tags"
SOURCE_NAME = "colab_real_corpus"
PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "real_prompts.jsonl"
EVAL_PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "real_eval_prompts.jsonl"
PROBLEM_METADATA_JSONL = PROJECT_ROOT / "data" / "real_problem_metadata.jsonl"
NUM_EVAL_SAMPLES = 8

if REAL_MODE_PRESET == "public_gsm8k_qwen25_math_1p5b":
    if is_placeholder(MODEL_ID):
        MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"
    SOURCE_NAME = "567-labs/gsm8k"
    PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "runs" / "colab_real_session" / "public_benchmark" / "prompt_manifest.jsonl"
    EVAL_PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "runs" / "colab_real_session" / "public_benchmark" / "eval_manifest.jsonl"
    PROBLEM_METADATA_JSONL = PROJECT_ROOT / "runs" / "colab_real_session" / "public_benchmark" / "problem_metadata.jsonl"
    if NUM_EVAL_SAMPLES == 8:
        NUM_EVAL_SAMPLES = 4

REAL_MODE_ISSUES = collect_real_mode_issues()

if EXPERIMENT_MODE == "starter":
    RESOLVED_EXPERIMENT_MODE = "starter"
elif EXPERIMENT_MODE == "real":
    RESOLVED_EXPERIMENT_MODE = "real"
else:
    RESOLVED_EXPERIMENT_MODE = "real" if not REAL_MODE_ISSUES else "starter"

if RESOLVED_EXPERIMENT_MODE == "starter":
    RUN_NAME = "colab_starter_fixture"
    MODEL_ID = "fixture_replay_bundle"
    RAW_INPUT_PATH = ""
    RAW_INPUT_FORMAT = "auto"
    PROBLEM_ID_FIELD = "problem_id"
    PROMPT_FIELD = "prompt_text"
    ANSWER_FIELD = "gold_answer"
    TOPIC_FIELD = "topic"
    DIFFICULTY_FIELD = "difficulty"
    TAGS_FIELD = "tags"
    SOURCE_NAME = "fixture_starter"
    PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "fixture_unlabeled_prompts.jsonl"
    EVAL_PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "fixture_eval_samples_a0.jsonl"
    PROBLEM_METADATA_JSONL = PROJECT_ROOT / "data" / "fixture_problem_metadata.jsonl"
    NUM_EVAL_SAMPLES = 4
else:
    RUN_NAME = "colab_real_session"

RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
BUNDLE_DIR = RUN_ROOT / "bundle"
LADDER_ROOT = RUN_ROOT / "starter_fixture_ladder"

WANDB_ENABLED = True
WANDB_SECRET_NAME = "WANDB_API_KEY"
WANDB_PROJECT = "SSD_AIMO3"
WANDB_ENTITY = ""
WANDB_MODE = "online"
WANDB_ARTIFACT_MAX_FILE_SIZE_MB = 25
WANDB_TAGS = ["colab", "notebook", RESOLVED_EXPERIMENT_MODE]
if REAL_MODE_PRESET and RESOLVED_EXPERIMENT_MODE == "real":
    WANDB_TAGS.append(REAL_MODE_PRESET)
WANDB_NOTES = f"SSD_AIMO3 Colab {RESOLVED_EXPERIMENT_MODE} session"

print(RUN_ROOT)

In [ ]:
# If this Colab session does not already have the repo checked out, clone it now.

ensure_repo(REPO_URL, REPO_REF)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

In [ ]:
# Resolve the run path. Auto mode now prefers the public real benchmark path and falls back to the fixture ladder only when real mode is not ready.

if EXPERIMENT_MODE == "real" and REAL_MODE_ISSUES:
    raise RuntimeError("Real mode is not ready:\n- " + "\n- ".join(REAL_MODE_ISSUES))

if EXPERIMENT_MODE == "auto" and REAL_MODE_ISSUES:
    print("Auto mode: real inputs are not ready yet, so this run will use the self-contained starter ladder.")
    for issue in REAL_MODE_ISSUES:
        print(f"- {issue}")
else:
    print(f"Preflight OK. Requested {EXPERIMENT_MODE}, running {RESOLVED_EXPERIMENT_MODE}.")

In [ ]:
# In Colab, run this once per fresh session after the repo is present.
# If deps are already installed, you can skip or comment out the pip line.

run("python", "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements" / "colab-gpu.txt"))
if WANDB_ENABLED:
    NOTEBOOK_WANDB_RUN = configure_wandb(
        secret_name=WANDB_SECRET_NAME,
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        mode=WANDB_MODE,
        tags=WANDB_TAGS,
        notes=WANDB_NOTES,
    )
    log_wandb_scalars(
        NOTEBOOK_WANDB_RUN,
        "session",
        {
            "experiment_mode": RESOLVED_EXPERIMENT_MODE,
            "requested_experiment_mode": EXPERIMENT_MODE,
            "real_mode_preset": REAL_MODE_PRESET,
            "run_name": RUN_NAME,
            "model_id": MODEL_ID,
            "num_eval_samples": NUM_EVAL_SAMPLES,
        },
    )
run("python", str(PROJECT_ROOT / "scripts" / "colab_runtime_probe.py"), "--output-json", str(RUN_ROOT / "colab_runtime.json"))
runtime = read_json(RUN_ROOT / "colab_runtime.json")
log_wandb_scalars(NOTEBOOK_WANDB_RUN, "runtime", runtime)
if NOTEBOOK_WANDB_RUN is not None:
    NOTEBOOK_WANDB_RUN.summary.update(flatten_scalars("runtime", runtime))
print(json.dumps(runtime, indent=2))

In [ ]:
# In real mode, either bootstrap the public benchmark preset or normalize a custom source into the repo's prompt/eval/metadata contracts.

if RESOLVED_EXPERIMENT_MODE == "starter":
    print("Starter mode uses the built-in fixture manifests.")
elif REAL_MODE_PRESET:
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "prepare_public_math_benchmark.py"),
        "--preset",
        REAL_MODE_PRESET,
        "--output-dir",
        str(RUN_ROOT / "public_benchmark"),
        "--train-limit",
        str(REAL_PRESET_TRAIN_LIMIT),
        "--eval-limit",
        str(REAL_PRESET_EVAL_LIMIT),
        "--seed",
        str(REAL_PRESET_SEED),
    )
    public_summary = read_json(RUN_ROOT / "public_benchmark" / "benchmark_summary.json")
    print(json.dumps(public_summary, indent=2))
elif RAW_INPUT_PATH:
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "build_problem_manifests.py"),
        "--input-path",
        RAW_INPUT_PATH,
        "--input-format",
        RAW_INPUT_FORMAT,
        "--prompt-output-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-output-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--metadata-output-jsonl",
        str(PROBLEM_METADATA_JSONL),
        "--summary-json",
        str(RUN_ROOT / "manifest_summary.json"),
        "--problem-id-field",
        PROBLEM_ID_FIELD,
        "--prompt-field",
        PROMPT_FIELD,
        "--answer-field",
        ANSWER_FIELD,
        "--topic-field",
        TOPIC_FIELD,
        "--difficulty-field",
        DIFFICULTY_FIELD,
        "--tags-field",
        TAGS_FIELD,
        "--source-name",
        SOURCE_NAME,
    )
else:
    print("Using pre-existing real manifests.")

print(PROMPT_MANIFEST_JSONL)
print(EVAL_PROMPT_MANIFEST_JSONL)
print(PROBLEM_METADATA_JSONL)

In [ ]:
# Materialize a clean Colab config bundle from one parameter surface for real runs.

if RESOLVED_EXPERIMENT_MODE == "starter":
    print("Starter mode uses the fixture configs directly and skips bundle materialization.")
else:
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "materialize_colab_bundle.py"),
        "--output-dir",
        str(BUNDLE_DIR),
        "--model-id",
        MODEL_ID,
        "--prompt-manifest-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-prompt-manifest-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--problem-metadata-jsonl",
        str(PROBLEM_METADATA_JSONL),
    )
    bundle_manifest = read_json(BUNDLE_DIR / "bundle_manifest.json")
    print(json.dumps(bundle_manifest, indent=2))

## Run A0 and A1

The early goal is not leaderboard chasing. The goal is to establish whether the student has any directional lift over the strongest fair frozen baseline under the same evaluation contract.

In [ ]:
# A0 baseline generation + evaluation

if RESOLVED_EXPERIMENT_MODE == "starter":
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "run_validation_ladder.py"),
        "--output-dir",
        str(LADDER_ROOT),
    )
    print(read_json(LADDER_ROOT / "a0_eval" / "metrics.json"))
    print(read_json(LADDER_ROOT / "a1_train" / "training_plan.json"))
else:
    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a0.yaml"),
        "--input-jsonl", str(EVAL_PROMPT_MANIFEST_JSONL),
        "--num-samples", str(NUM_EVAL_SAMPLES),
        "--output-dir", str(RUN_ROOT / "a0_eval_generation"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a0.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a0_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a0_eval"),
    )

    # A1 self-sample generation + training launch
    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a1.yaml"),
        "--input-jsonl", str(PROMPT_MANIFEST_JSONL),
        "--output-dir", str(RUN_ROOT / "a1_self_samples"),
    )
    print(read_json(RUN_ROOT / "a1_self_samples" / "generation_metrics.json"))
    run(
        "python", str(PROJECT_ROOT / "scripts" / "train_ssd_math.py"),
        "--config", str(BUNDLE_DIR / "a1.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_self_samples" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a1_train"),
        "--launch",
    )

    print(read_json(RUN_ROOT / "a0_eval" / "metrics.json"))
    print(read_json(RUN_ROOT / "a1_train" / "training_plan.json"))

In [ ]:
# Re-materialize the bundle with the trained adapter path so the student can be evaluated honestly.

if RESOLVED_EXPERIMENT_MODE == "starter":
    print(read_json(LADDER_ROOT / "a1_eval" / "metrics.json"))
else:
    ADAPTER_PATH = RUN_ROOT / "a1_train" / "adapter"
    assert ADAPTER_PATH.exists(), f"Expected adapter at {ADAPTER_PATH}"

    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "materialize_colab_bundle.py"),
        "--output-dir",
        str(BUNDLE_DIR),
        "--model-id",
        MODEL_ID,
        "--prompt-manifest-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-prompt-manifest-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--problem-metadata-jsonl",
        str(PROBLEM_METADATA_JSONL),
        "--student-adapter-path",
        str(ADAPTER_PATH),
    )

    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a1_student_eval.yaml"),
        "--input-jsonl", str(EVAL_PROMPT_MANIFEST_JSONL),
        "--num-samples", str(NUM_EVAL_SAMPLES),
        "--output-dir", str(RUN_ROOT / "a1_eval_generation"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a1_student_eval.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a1_eval"),
    )

    print(read_json(RUN_ROOT / "a1_eval" / "metrics.json"))

In [ ]:
# Run A5 on the exact same A1 eval generations, then compare A0 vs A1 and A1 vs A5.

if RESOLVED_EXPERIMENT_MODE == "starter":
    a0_metrics = read_json(LADDER_ROOT / "a0_eval" / "metrics.json")
    a1_metrics = read_json(LADDER_ROOT / "a1_eval" / "metrics.json")
    a5_metrics = read_json(LADDER_ROOT / "a5_eval" / "metrics.json")
    a0_vs_a1 = read_json(LADDER_ROOT / "compare_a0_a1" / "comparison_summary.json")
    a1_vs_a5 = read_json(LADDER_ROOT / "compare_a1_a5" / "comparison_summary.json")
else:
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a5.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a5_eval"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "compare_eval_runs.py"),
        "--run-a-dir", str(RUN_ROOT / "a0_eval"),
        "--run-b-dir", str(RUN_ROOT / "a1_eval"),
        "--run-a-label", "a0_majority_vote",
        "--run-b-label", "a1_majority_vote",
        "--metadata-jsonl", str(PROBLEM_METADATA_JSONL),
        "--output-dir", str(RUN_ROOT / "compare_a0_a1"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "compare_eval_runs.py"),
        "--run-a-dir", str(RUN_ROOT / "a1_eval"),
        "--run-b-dir", str(RUN_ROOT / "a5_eval"),
        "--run-a-label", "a1_majority_vote",
        "--run-b-label", "a5_tropical_rerank",
        "--metadata-jsonl", str(PROBLEM_METADATA_JSONL),
        "--output-dir", str(RUN_ROOT / "compare_a1_a5"),
    )

    a0_metrics = read_json(RUN_ROOT / "a0_eval" / "metrics.json")
    a1_metrics = read_json(RUN_ROOT / "a1_eval" / "metrics.json")
    a5_metrics = read_json(RUN_ROOT / "a5_eval" / "metrics.json")
    a0_vs_a1 = read_json(RUN_ROOT / "compare_a0_a1" / "comparison_summary.json")
    a1_vs_a5 = read_json(RUN_ROOT / "compare_a1_a5" / "comparison_summary.json")

def accuracy_metric(metrics):
    return metrics.get("exact_final_answer_accuracy", metrics.get("accuracy"))

a0_accuracy = accuracy_metric(a0_metrics)
a1_accuracy = accuracy_metric(a1_metrics)
a5_accuracy = accuracy_metric(a5_metrics)

decision_summary = {
    "run_root": str(RUN_ROOT),
    "requested_experiment_mode": EXPERIMENT_MODE,
    "experiment_mode": RESOLVED_EXPERIMENT_MODE,
    "real_mode_preset": REAL_MODE_PRESET,
    "model_id": MODEL_ID,
    "a0_accuracy": a0_accuracy,
    "a1_accuracy": a1_accuracy,
    "a5_accuracy": a5_accuracy,
    "a1_minus_a0_accuracy": (a1_accuracy or 0.0) - (a0_accuracy or 0.0),
    "a5_minus_a1_accuracy": (a5_accuracy or 0.0) - (a1_accuracy or 0.0),
    "a0_vs_a1_discordant_pairs": a0_vs_a1.get("discordant_pairs"),
    "a0_vs_a1_sign_test_pvalue": a0_vs_a1.get("paired_sign_test_pvalue_two_sided"),
    "a1_vs_a5_discordant_pairs": a1_vs_a5.get("discordant_pairs"),
    "a1_vs_a5_sign_test_pvalue": a1_vs_a5.get("paired_sign_test_pvalue_two_sided"),
    "thesis_status": "starter_fixture_complete" if RESOLVED_EXPERIMENT_MODE == "starter" else "undecided_review_outputs",
}
write_json(RUN_ROOT / "decision_summary.json", decision_summary)
log_wandb_scalars(NOTEBOOK_WANDB_RUN, "decision", decision_summary)
if NOTEBOOK_WANDB_RUN is not None:
    NOTEBOOK_WANDB_RUN.summary.update(flatten_scalars("decision", decision_summary))
    NOTEBOOK_WANDB_RUN.finish()

print("A0 metrics:\n", json.dumps(a0_metrics, indent=2))
print("A1 metrics:\n", json.dumps(a1_metrics, indent=2))
print("A5 metrics:\n", json.dumps(a5_metrics, indent=2))
print("A0 vs A1:\n", json.dumps(a0_vs_a1, indent=2))
print("A1 vs A5:\n", json.dumps(a1_vs_a5, indent=2))
print("Decision summary:\n", json.dumps(decision_summary, indent=2))

## Decision pass

Use the outputs above to write down an explicit conclusion before changing prompts or budgets again:

- `Proceed`: A1 beats A0 under a fair contract and the gain looks robust enough to merit more budget.
- `Proceed conditionally`: A1 is weak but A5 or a later controlled extension adds value.
- `Do not proceed`: gains vanish, depend on bad extraction, or fail under fair paired comparison.

Red-team questions:

- Did we accidentally give the student a different inference budget?
- Did extraction success change enough to explain the gain?
- Are the discordant-pair counts large enough to trust the direction?
- Which topic slices improved, and which did not?
- What is the cheapest next experiment that would most strongly falsify the current interpretation?